# Image Generation — Diffusion Models Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Noise schedule

In [ ]:
```python

import torch

def linear_beta_schedule(T=1000, beta_start=1e-4, beta_end=2e-2):

    return torch.linspace(beta_start, beta_end, T)

def precompute_schedule(betas):

    alphas = 1.0 - betas

    alphas_cumprod = torch.cumprod(alphas, dim=0)

    return {

        "betas": betas,

        "alphas": alphas,

        "alphas_cumprod": alphas_cumprod,

        "sqrt_alphas_cumprod": torch.sqrt(alphas_cumprod),

        "sqrt_one_minus_alphas_cumprod": torch.sqrt(1.0 - alphas_cumprod),

        "sqrt_recip_alphas": torch.sqrt(1.0 / alphas),

    }

schedule = precompute_schedule(linear_beta_schedule(T=1000))

In [ ]:
```

Precompute once, gather by index during training and sampling.

### Step 2: Forward diffusion (q_sample)

In [ ]:
```python

def q_sample(x0, t, noise, schedule):

    sqrt_a = schedule["sqrt_alphas_cumprod"][t].view(-1, 1, 1, 1)

    sqrt_one_minus_a = schedule["sqrt_one_minus_alphas_cumprod"][t].view(-1, 1, 1, 1)

    return sqrt_a * x0 + sqrt_one_minus_a * noise

In [ ]:
```

One-line closed form. `t` is a batch of timesteps, one per image in the batch.

### Step 3: A tiny time-conditioned U-Net

In [ ]:
```python

import torch.nn as nn

import torch.nn.functional as F

import math

def timestep_embedding(t, dim=64):

    half = dim // 2

    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)

    args = t[:, None].float() * freqs[None]

    emb = torch.cat([args.sin(), args.cos()], dim=-1)

    return emb

class TinyUNet(nn.Module):

    def __init__(self, img_channels=3, base=32, t_dim=64):

        super().__init__()

        self.t_mlp = nn.Sequential(

            nn.Linear(t_dim, base * 4),

            nn.SiLU(),

            nn.Linear(base * 4, base * 4),

        )

        self.t_dim = t_dim

        self.enc1 = nn.Conv2d(img_channels, base, 3, padding=1)

        self.enc2 = nn.Conv2d(base, base * 2, 4, stride=2, padding=1)

        self.mid = nn.Conv2d(base * 2, base * 2, 3, padding=1)

        self.dec1 = nn.ConvTranspose2d(base * 2, base, 4, stride=2, padding=1)

        self.dec2 = nn.Conv2d(base * 2, img_channels, 3, padding=1)

        self.time_proj = nn.Linear(base * 4, base * 2)

    def forward(self, x, t):

        t_emb = timestep_embedding(t, self.t_dim)

        t_emb = self.t_mlp(t_emb)

        t_proj = self.time_proj(t_emb)[:, :, None, None]

        h1 = F.silu(self.enc1(x))

        h2 = F.silu(self.enc2(h1)) + t_proj

        h3 = F.silu(self.mid(h2))

        d1 = F.silu(self.dec1(h3))

        d2 = torch.cat([d1, h1], dim=1)

        return self.dec2(d2)

In [ ]:
```

Two-level U-Net with time conditioning injected at the bottleneck. Scale up the depth and width for real images.

### Step 4: Training loop

In [ ]:
```python

def train_step(model, x0, schedule, optimizer, device, T=1000):

    model.train()

    x0 = x0.to(device)

    bs = x0.size(0)

    t = torch.randint(0, T, (bs,), device=device)

    noise = torch.randn_like(x0)

    x_t = q_sample(x0, t, noise, schedule)

    pred = model(x_t, t)

    loss = F.mse_loss(pred, noise)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    return loss.item()

In [ ]:
```

That is the entire training loop. No GAN game, no specialised loss, one MSE call.

### Step 5: Sampler (DDPM)

In [ ]:
```python

@torch.no_grad()

def sample(model, schedule, shape, T=1000, device="cpu"):

    model.eval()

    x = torch.randn(shape, device=device)

    betas = schedule["betas"].to(device)

    sqrt_one_minus_a = schedule["sqrt_one_minus_alphas_cumprod"].to(device)

    sqrt_recip_alphas = schedule["sqrt_recip_alphas"].to(device)

    for t in reversed(range(T)):

        t_batch = torch.full((shape[0],), t, dtype=torch.long, device=device)

        eps = model(x, t_batch)

        coef = betas[t] / sqrt_one_minus_a[t]

        mean = sqrt_recip_alphas[t] * (x - coef * eps)

        if t > 0:

            x = mean + torch.sqrt(betas[t]) * torch.randn_like(x)

        else:

            x = mean

    return x

In [ ]:
```

1000 forward passes to produce one batch of samples. In real code you would swap this for a DDIM 50-step sampler.

### Step 6: DDIM sampler (deterministic, ~20x faster)

In [ ]:
```python

@torch.no_grad()

def sample_ddim(model, schedule, shape, steps=50, T=1000, device="cpu", eta=0.0):

    model.eval()

    x = torch.randn(shape, device=device)

    alphas_cumprod = schedule["alphas_cumprod"].to(device)

    ts = torch.linspace(T - 1, 0, steps + 1).long()

    for i in range(steps):

        t = ts[i]

        t_prev = ts[i + 1]

        t_batch = torch.full((shape[0],), t, dtype=torch.long, device=device)

        eps = model(x, t_batch)

        a_t = alphas_cumprod[t]

        a_prev = alphas_cumprod[t_prev] if t_prev >= 0 else torch.tensor(1.0, device=device)

        x0_pred = (x - torch.sqrt(1 - a_t) * eps) / torch.sqrt(a_t)

        sigma = eta * torch.sqrt((1 - a_prev) / (1 - a_t) * (1 - a_t / a_prev))

        dir_xt = torch.sqrt(1 - a_prev - sigma ** 2) * eps

        noise = sigma * torch.randn_like(x) if eta > 0 else 0

        x = torch.sqrt(a_prev) * x0_pred + dir_xt + noise

    return x

In [ ]:
```

`eta=0` is fully deterministic (same noise input always produces the same output). `eta=1` recovers DDPM.

## Exercises

In [ ]:
1. **(Easy)** Visualise the forward process: take one image and plot `x_t` at `t in [0, 100, 250, 500, 750, 1000]`. Verify that `x_1000` looks like pure Gaussian noise.
2. **(Medium)** Train the TinyUNet on the synthetic-circles dataset for 20 epochs and sample 16 circles. Compare DDPM (1000 steps) and DDIM (50 steps) sampling — do they produce similar images from the same noise seed?
3. **(Hard)** Implement a cosine noise schedule (Nichol & Dhariwal, 2021): `alpha_bar_t = cos^2((t/T + s) / (1 + s) * pi / 2)`. Train the same model with linear and cosine schedules and show that cosine gives better samples at low step counts.